# Notebook 1: Tokenisation
### What happens before the model sees a single word?

**No external models needed.** Uses `tiktoken` (OpenAI's tokeniser library) + pure Python.

---
Topics covered:
1. What is a token?
2. Tokenising English, banking terms, Hindi, symbols
3. Token count vs word count
4. Cost estimation from token counts
5. Why tokeniser choice matters across models

In [1]:
# Install tiktoken if not available
# pip install tiktoken
try:
    import tiktoken
    print("tiktoken available ✓")
except ImportError:
    print("Run: pip install tiktoken")

tiktoken available ✓


## 1. What is a Token?
A token is a sub-word unit — roughly 3-4 characters on average in English.
The tokeniser splits raw text into integer IDs before the model sees anything.

In [2]:
import tiktoken

# GPT-4 / GPT-3.5 tokeniser (cl100k_base)
enc = tiktoken.get_encoding("cl100k_base")

def show_tokens(text, enc=enc):
    tokens = enc.encode(text)
    decoded = [enc.decode([t]) for t in tokens]
    print(f"Text    : {text!r}")
    print(f"Token IDs : {tokens}")
    print(f"Token pieces: {decoded}")
    print(f"Count   : {len(tokens)} tokens\n")

# Simple English
show_tokens("bank")
show_tokens("banking")

Text    : 'bank'
Token IDs : [17469]
Token pieces: ['bank']
Count   : 1 tokens

Text    : 'banking'
Token IDs : [17469, 287]
Token pieces: ['bank', 'ing']
Count   : 2 tokens



In [3]:
# Banking domain vocabulary — from the deck
banking_terms = [
    "bank",
    "undercollateralised",
    "SWIFT",
    "KYC",
    "CIBIL",
    "Non-Performing Asset",
    "FOIR",
    "RBI Master Directions",
    "EMI",
    "collateralised debt obligation",
]

print(f"{'Term':<35} {'Tokens':>6} {'IDs'}")
print("-" * 80)
for term in banking_terms:
    tokens = enc.encode(term)
    pieces = [enc.decode([t]) for t in tokens]
    print(f"{term:<35} {len(tokens):>6}   {pieces}")

Term                                Tokens IDs
--------------------------------------------------------------------------------
bank                                     1   ['bank']
undercollateralised                      4   ['under', 'coll', 'ateral', 'ised']
SWIFT                                    2   ['SW', 'IFT']
KYC                                      2   ['KY', 'C']
CIBIL                                    3   ['C', 'IB', 'IL']
Non-Performing Asset                     5   ['Non', '-', 'Perform', 'ing', ' Asset']
FOIR                                     2   ['FO', 'IR']
RBI Master Directions                    4   ['R', 'BI', ' Master', ' Directions']
EMI                                      2   ['EM', 'I']
collateralised debt obligation           5   ['coll', 'ateral', 'ised', ' debt', ' obligation']


In [4]:
# The rupee symbol and non-ASCII text tokenises differently
print("=== Currency & Special Characters ===")
show_tokens("₹50,00,000")
show_tokens("$50,000")
show_tokens("€50,000")

print("=== SWIFT Transfer Example from Deck ===")
show_tokens("SWIFT transfer to HDFC account")

=== Currency & Special Characters ===
Text    : '₹50,00,000'
Token IDs : [16275, 117, 1135, 11, 410, 11, 931]
Token pieces: ['�', '�', '50', ',', '00', ',', '000']
Count   : 7 tokens

Text    : '$50,000'
Token IDs : [3, 1135, 11, 931]
Token pieces: ['$', '50', ',', '000']
Count   : 4 tokens

Text    : '€50,000'
Token IDs : [15406, 1135, 11, 931]
Token pieces: ['€', '50', ',', '000']
Count   : 4 tokens

=== SWIFT Transfer Example from Deck ===
Text    : 'SWIFT transfer to HDFC account'
Token IDs : [17268, 14627, 8481, 311, 81340, 34, 2759]
Token pieces: ['SW', 'IFT', ' transfer', ' to', ' HDF', 'C', ' account']
Count   : 7 tokens



## 2. Token Count vs Word Count
Engineers often assume 1 word = 1 token. This is wrong and leads to cost miscalculations.

In [5]:
sample_texts = {
    "Simple English sentence": "The customer submitted a loan application yesterday.",
    "Banking legal clause": "The borrower shall maintain a Fixed Obligation to Income Ratio (FOIR) not exceeding 55% throughout the tenure of the loan.",
    "RBI-style circular text": "All Scheduled Commercial Banks (SCBs) are hereby directed to maintain a Capital to Risk-weighted Assets Ratio (CRAR) of not less than 11.5% as on March 31, 2026.",
    "Python code snippet": "def calculate_emi(principal, rate, tenure):\n    r = rate / (12 * 100)\n    return principal * r * (1+r)**tenure / ((1+r)**tenure - 1)",
    "Mixed English-Hindi": "Customer ka CIBIL score 712 hai aur income verification pending hai.",
}

print(f"{'Description':<30} {'Words':>6} {'Tokens':>7} {'Ratio':>8}")
print("-" * 60)
for desc, text in sample_texts.items():
    words = len(text.split())
    tokens = len(enc.encode(text))
    ratio = tokens / words
    print(f"{desc:<30} {words:>6} {tokens:>7} {ratio:>8.2f}x")

Description                     Words  Tokens    Ratio
------------------------------------------------------------
Simple English sentence             7       8     1.14x
Banking legal clause               20      27     1.35x
RBI-style circular text            27      44     1.63x
Python code snippet                21      45     2.14x
Mixed English-Hindi                11      15     1.36x


## 3. Document Size → Token Count → Cost
Real cost estimation for banking document workflows

In [6]:
# Simulate a 10-page policy document
sample_page = """
CHAPTER 3: CREDIT RISK MANAGEMENT FRAMEWORK

3.1 Credit Appraisal Process
All credit proposals above INR 10 lakhs shall be processed through the centralized 
credit appraisal system. The relationship manager shall submit a Credit Appraisal 
Memorandum (CAM) containing the following mandatory fields: applicant details, 
income verification, bureau report analysis, collateral valuation, and risk rating.

3.2 FOIR Calculation
The Fixed Obligation to Income Ratio shall be calculated as the sum of all monthly 
fixed obligations (including the proposed EMI) divided by the gross monthly income 
of the applicant. The maximum permissible FOIR for salaried individuals is 55% and 
for self-employed individuals is 50%.

3.3 CIBIL Score Thresholds
Minimum acceptable CIBIL score: 700 for retail loans, 650 for MSME loans.
Applicants with scores below 650 shall be declined at the pre-screening stage 
without further processing.
"""

tokens_per_page = len(enc.encode(sample_page))
print(f"Tokens per page (estimated): {tokens_per_page}")

# Cost table for different document sizes
print("\n=== Cost Estimation Table ===")
print(f"{'Document':<25} {'Pages':>6} {'Tokens':>10} {'GPT-5 Input Cost':>18} {'Claude Opus Cost':>18}")
print("-" * 82)

# Pricing from deck (per million tokens)
gpt5_input_per_m = 10.0  # $10/M
claude_opus_input_per_m = 5.0  # $5/M

docs = [
    ("1-page complaint", 1),
    ("KYC document set", 5),
    ("Loan agreement", 20),
    ("RBI circular", 15),
    ("Annual report", 150),
    ("NCLT order", 300),
]

for doc_name, pages in docs:
    tokens = pages * tokens_per_page
    gpt5_cost = (tokens / 1_000_000) * gpt5_input_per_m
    claude_cost = (tokens / 1_000_000) * claude_opus_input_per_m
    print(f"{doc_name:<25} {pages:>6} {tokens:>10,} {f'${gpt5_cost:.4f}':>18} {f'${claude_cost:.4f}':>18}")

print("\nNote: Output tokens cost more. Budget 30% extra for model responses.")

Tokens per page (estimated): 201

=== Cost Estimation Table ===
Document                   Pages     Tokens   GPT-5 Input Cost   Claude Opus Cost
----------------------------------------------------------------------------------
1-page complaint               1        201            $0.0020            $0.0010
KYC document set               5      1,005            $0.0100            $0.0050
Loan agreement                20      4,020            $0.0402            $0.0201
RBI circular                  15      3,015            $0.0301            $0.0151
Annual report                150     30,150            $0.3015            $0.1507
NCLT order                   300     60,300            $0.6030            $0.3015

Note: Output tokens cost more. Budget 30% extra for model responses.


In [7]:
# Scale to production: what happens at 10,000 calls/day?
print("=== Production Scale Cost (per day) ===")
system_prompt_tokens = 500   # typical system prompt
doc_tokens = 3000             # avg document
output_tokens = 300           # avg response
calls_per_day = 10_000

input_tokens_per_day = (system_prompt_tokens + doc_tokens) * calls_per_day
output_tokens_per_day = output_tokens * calls_per_day

gpt5_output_per_m = 30.0  # $30/M output
claude_output_per_m = 20.0

gpt5_daily = (input_tokens_per_day / 1e6 * gpt5_input_per_m) + (output_tokens_per_day / 1e6 * gpt5_output_per_m)
claude_daily = (input_tokens_per_day / 1e6 * claude_opus_input_per_m) + (output_tokens_per_day / 1e6 * claude_output_per_m)

print(f"Calls/day          : {calls_per_day:,}")
print(f"Input tokens/day   : {input_tokens_per_day:,}")
print(f"Output tokens/day  : {output_tokens_per_day:,}")
print(f"GPT-5 daily cost   : ${gpt5_daily:.2f}  (${gpt5_daily*30:.2f}/month)")
print(f"Claude Opus daily  : ${claude_daily:.2f}  (${claude_daily*30:.2f}/month)")
print(f"\nKey insight: A 2000-token system prompt × 10K calls = {2000*10000:,} tokens/day in system prompt alone!")

=== Production Scale Cost (per day) ===
Calls/day          : 10,000
Input tokens/day   : 35,000,000
Output tokens/day  : 3,000,000
GPT-5 daily cost   : $440.00  ($13200.00/month)
Claude Opus daily  : $235.00  ($7050.00/month)

Key insight: A 2000-token system prompt × 10K calls = 20,000,000 tokens/day in system prompt alone!


## 4. Manual BPE — How Tokenisation Actually Works
Byte Pair Encoding (BPE) is the algorithm behind tiktoken. Here's the core loop.

In [8]:
from collections import Counter

def get_pairs(vocab):
    """Get all adjacent pairs in the vocabulary"""
    pairs = Counter()
    for word, freq in vocab.items():
        symbols = word.split()
        for i in range(len(symbols) - 1):
            pairs[(symbols[i], symbols[i+1])] += freq
    return pairs

def merge_vocab(pair, vocab):
    """Merge the most frequent pair"""
    new_vocab = {}
    bigram = ' '.join(pair)
    replacement = ''.join(pair)
    for word in vocab:
        new_word = word.replace(bigram, replacement)
        new_vocab[new_word] = vocab[word]
    return new_vocab

# Simple banking corpus — character-level start
corpus = [
    "loan", "loan", "loan", "loans",
    "lower", "low", "lowest",
    "bank", "bank", "banking", "banker",
    "risk", "risks", "risky",
]

# Start: each word split into chars with end-of-word marker
vocab = Counter()
for word in corpus:
    vocab[' '.join(list(word)) + ' </w>'] += 1

print("Initial vocabulary (character level):")
for word, freq in sorted(vocab.items(), key=lambda x: -x[1]):
    print(f"  {freq}x  {word}")

print("\n--- Running 6 BPE merge steps ---")
for step in range(6):
    pairs = get_pairs(vocab)
    if not pairs:
        break
    best = max(pairs, key=pairs.get)
    vocab = merge_vocab(best, vocab)
    print(f"\nStep {step+1}: Merged {best[0]!r} + {best[1]!r} → {''.join(best)!r}  (freq={pairs[best]})")
    for word, freq in sorted(vocab.items(), key=lambda x: -x[1]):
        print(f"         {freq}x  {word}")

Initial vocabulary (character level):
  3x  l o a n </w>
  2x  b a n k </w>
  1x  l o a n s </w>
  1x  l o w e r </w>
  1x  l o w </w>
  1x  l o w e s t </w>
  1x  b a n k i n g </w>
  1x  b a n k e r </w>
  1x  r i s k </w>
  1x  r i s k s </w>
  1x  r i s k y </w>

--- Running 6 BPE merge steps ---

Step 1: Merged 'a' + 'n' → 'an'  (freq=8)
         3x  l o an </w>
         2x  b an k </w>
         1x  l o an s </w>
         1x  l o w e r </w>
         1x  l o w </w>
         1x  l o w e s t </w>
         1x  b an k i n g </w>
         1x  b an k e r </w>
         1x  r i s k </w>
         1x  r i s k s </w>
         1x  r i s k y </w>

Step 2: Merged 'l' + 'o' → 'lo'  (freq=7)
         3x  lo an </w>
         2x  b an k </w>
         1x  lo an s </w>
         1x  lo w e r </w>
         1x  lo w </w>
         1x  lo w e s t </w>
         1x  b an k i n g </w>
         1x  b an k e r </w>
         1x  r i s k </w>
         1x  r i s k s </w>
         1x  r i s k y </w>

Step 3: Merged

## 5. The "Lost in the Middle" Phenomenon
Critical prompt engineering insight from the deck: attention is NOT uniform across all tokens.

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend
import matplotlib.pyplot as plt

# Simulate attention weight distribution across a long document
# Research shows models pay more attention to beginning and end
def simulate_attention_weights(n_positions, sink_strength=3.0):
    """Approximate U-shaped attention distribution (primacy + recency bias)"""
    positions = np.linspace(0, 1, n_positions)
    # U-shaped: high at start (attention sink) and end (recency)
    weights = np.exp(-sink_strength * positions) + 0.3 * np.exp(-sink_strength * (1 - positions))
    weights += np.random.uniform(0, 0.05, n_positions)  # noise
    weights = weights / weights.sum()
    return weights

n = 200  # positions in a long document
weights = simulate_attention_weights(n)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Attention distribution
axes[0].fill_between(range(n), weights, alpha=0.6, color='steelblue')
axes[0].axvspan(0, 20, alpha=0.2, color='green', label='Beginning (high attention)')
axes[0].axvspan(80, 120, alpha=0.2, color='red', label='Middle (low attention - DANGER ZONE)')
axes[0].axvspan(180, 200, alpha=0.2, color='orange', label='End (moderate attention)')
axes[0].set_title('Attention Weight Distribution\nacross a Long Document', fontsize=13)
axes[0].set_xlabel('Token Position')
axes[0].set_ylabel('Attention Weight')
axes[0].legend(fontsize=9)

# Plot 2: Practical implication for prompt design
prompt_sections = ['System\nInstructions', 'Context\nDoc Part 1', 'Context\nDoc Part 2', 
                   'Context\nDoc Part 3', 'User\nQuestion']
avg_attention = [0.35, 0.18, 0.08, 0.14, 0.25]
colors = ['green', 'orange', 'red', 'orange', 'green']
bars = axes[1].bar(prompt_sections, avg_attention, color=colors, alpha=0.7, edgecolor='black')
axes[1].set_title('Average Attention by Prompt Section\n("Lost in the Middle" Effect)', fontsize=13)
axes[1].set_ylabel('Relative Attention Weight')
for bar, val in zip(bars, avg_attention):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{val:.0%}', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/attention_distribution.png', dpi=150, bbox_inches='tight')
plt.close()
print("Chart saved.")

print("\n=== Practical Rule ===")
print("Put CRITICAL instructions at the START and END of your prompt.")
print("Information buried in the middle of a long prompt gets less attention.")
print("This is why the deck says: place key instructions at beginning AND end.")

## 6. Token Vocabulary — What the Model 'Knows'
Exploring what single-token vs multi-token words tell us about model knowledge

In [ ]:
# Compare banking jargon token efficiency across different domains
word_groups = {
    "Common English": ["the", "bank", "loan", "risk", "money", "account", "payment"],
    "Banking Technical": ["CIBIL", "FOIR", "KYC", "AML", "NBFC", "PMLA", "NACH"],
    "Legal/Regulatory": ["collateralised", "hypothecation", "subrogation", "indemnification"],
    "Indian Specific": ["lakh", "crore", "Aadhaar", "PAN", "Nifty", "Sensex"],
    "Tech Terms": ["transformer", "tokenisation", "embedding", "hallucination", "RAG"],
}

print(f"{'Group':<20} {'Word':<25} {'Tokens':>7} {'Pieces'}")
print("-" * 75)
for group, words in word_groups.items():
    for word in words:
        tokens = enc.encode(word)
        pieces = [enc.decode([t]) for t in tokens]
        print(f"{group:<20} {word:<25} {len(tokens):>7}   {pieces}")
    print()

print("\nKey insight: Multi-token words cost more AND the model may be less confident")
print("about rare banking acronyms since it saw fewer examples during pre-training.")

## Summary: What You Learned

| Concept | Key Takeaway |
|---------|-------------|
| Token ≠ Word | Avg 1.3-1.5x tokens per word in English banking text |
| Rare words | Split into more tokens — cost more, model less confident |
| ₹ symbol | 2-3 tokens — non-ASCII is expensive |
| BPE algorithm | Learns frequent pairs from training corpus |
| Lost in middle | Put critical instructions at prompt START and END |
| Cost math | Always profile your domain vocabulary before estimating costs |